# Adım 5: Feature Engineering
Silver katmanını oluştur.

### Özelliklerin İş Mantığı (İş Kuralları ve Seçim Nedenleri)
- **pickup_hour (Saat)**: Taksi ücretleri saat dilimine göre büyük değişiklik gösterir (gece tarifesi, sabah yoğunluğu vb.). Bu nedenle günün saatini ayrı bir değişken olarak çıkardık.
- **day_of_week (Haftanın Günü)**: Hafta içi iş trafiği ile hafta sonu eğlence/dinlenme trafiği farklıdır.
- **is_weekend (Hafta Sonu mu?)**: Cumartesi ve Pazar günlerini 1, diğer günleri 0 yaparak modelin hafta sonu kalıplarını daha rahat öğrenmesini sağladık.
- **trip_duration (Yolculuk Süresi)**: Mesafe aynı olsa bile trafikte geçen süre ücreti (bekleme ücreti gibi) etkiler. Bu yüzden pickup ve dropoff zaman farkından süreyi dakika cinsinden hesapladık.
- **is_airport (Havalimanı mı?)**: 132 (JFK) ve 138 (LaGuardia) ID'li bölgeler havalimanıdır. Havalimanı yolculuklarında özel tarifeler veya ek ücretler olabildiğinden bunu binary (0-1) bir flag olarak belirttik.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, hour, dayofweek, when

spark = SparkSession.builder \
    .appName("FE") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

df = spark.read.format("delta").load("/app/delta/taxi_bronze")

df_clean = df.filter((col("fare_amount") > 0) & (col("trip_distance") > 0))

df_features = df_clean \
    .withColumn("pickup_ts", col("tpep_pickup_datetime").cast("timestamp")) \
    .withColumn("dropoff_ts", col("tpep_dropoff_datetime").cast("timestamp")) \
    .withColumn("pickup_hour", hour(col("pickup_ts"))) \
    .withColumn("day_of_week", dayofweek(col("pickup_ts"))) \
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0)) \
    .withColumn("trip_duration", (unix_timestamp(col("dropoff_ts")) - unix_timestamp(col("pickup_ts"))) / 60) \
    .withColumn("is_airport", when(col("PULocationID").isin(132, 138) | col("DOLocationID").isin(132, 138), 1).otherwise(0))

df_final = df_features.filter(col("trip_duration") > 0)

df_final.write.format("delta").mode("overwrite").save("/app/delta/taxi_silver")
print("Feature Engineering tamamlandı ve Silver katmanına yazıldı.")


Feature Engineering tamamlandı ve Silver katmanına yazıldı.
